In [1]:
import math_toolkit
math_toolkit.notebook.activate()

# ExprTransform

Use one transform definition as a normal function, a `>>` pipeline step, or a fluent chain.

`factor(simplify(expand(expr)))`  
`expr >> expand >> simplify >> factor`  
`ExprTransform(expr).expand().simplify().factor().Done()`

In [2]:
expr = (x + 1)**2 + sin(x)**2 + cos(x)**2
transformed = expr >> expand >> simplify >> factor

expr @Eq@ transformed

       2      2         2       2          
(x + 1)  + sin (x) + cos (x) = x  + 2⋅x + 2

## Core behavior

`ExprTransform` is a small wrapper for step-by-step symbolic transformation. The same operation object powers all three public styles: call it as a function, send an expression through it with `>>`, or use it as a method on an `ExprTransform` chain.

Direct transforms, such as `expand`, `simplify`, and `factor`, need only the expression. Argument-taking transforms, such as `subs`, `diff`, `collect`, and `evalf`, capture their extra arguments before the expression reaches them in a pipeline.

In [ ]:
expr = (x + 1)**3 - (x - 1)**3

function_style = factor(simplify(expand(expr)))
pipeline_style = expr >> expand >> simplify >> factor
fluent_style = ExprTransform(expr).expand().simplify().factor().Done()
transform_rows = [
    {"style": "function calls", "result": function_style},
    {"style": "pipeline", "result": pipeline_style},
    {"style": "fluent object", "result": fluent_style},
]

md("""Function calls, pipeline syntax, and fluent syntax all reach the same factored form.

{table(header=["style", "result"])}{{ transform_rows }}
""")

## Common patterns

**Substitute late.** Keep algebraic cleanup readable, then capture substitutions as a pipeline stage when the expression is ready.

In [4]:
energy = (q + x)**2 + (p - x)**2
point_value = energy >> expand >> collect(x) >> subs({x: 2, p: 3, q: -1})

energy @Eq@ point_value

       2          2    
(p - x)  + (q + x)  = 2

**Differentiate and continue.** Argument-taking transforms compose with direct transforms without changing notation midstream.

In [5]:
potential = (x**2 + 1) * exp(x)
force_shape = potential >> diff(x) >> expand >> collect(exp(x))

diff(potential, x) @Eq@ force_shape

     x   ⎛ 2    ⎞  x   ⎛ 2          ⎞  x
2⋅x⋅ℯ  + ⎝x  + 1⎠⋅ℯ  = ⎝x  + 2⋅x + 1⎠⋅ℯ 

**Fluent chains keep a longer edit in one object.** Use `.Done()` only when you want the raw SymPy expression back.

In [6]:
signal = sin(x)**2 + cos(x)**2 + (x + 1)**2

ExprTransform(signal).expand().simplify().subs({x: 2}).evalf(20).Done()

10.000000000000000000

**Expand named definitions.** `unhold` is shorthand for `rewrite("unhold")`, and it forwards rewrite options such as `deep`.

In [7]:
Step = Function("Step")(x) @EqDef@ (x + 1)
nested = Step(Step(x))
deep_expansion = nested
shallow_expansion = nested(deep=False)

md("""Deep expansion rewrites both nested calls; shallow expansion rewrites only the outer call.

- Deep expansion: {{ deep_expansion }}
- Shallow expansion: {{ shallow_expansion }}
""")

Deep expansion rewrites both nested calls; shallow expansion rewrites only the outer call.

- Deep expansion: $x + 2$
- Shallow expansion: $\mathrm{Step}(x) + 1$


## Options and Details

**Register a project transform.** `make_transform_op` turns any expression-first callable into an operation that has the same function, pipeline, and fluent surfaces.

In [ ]:
def shift_x(expr, amount):
    return expr.subs(x, x + amount)

shift_x = make_transform_op(shift_x, mode="factory")
shift_rows = [
    {"style": "function call", "result": shift_x(sin(x), pi / 2) >> simplify},
    {"style": "pipeline", "result": sin(x) >> shift_x(pi / 2) >> simplify},
    {"style": "fluent object", "result": ExprTransform(sin(x)).shift_x(pi / 2).simplify().Done()},
]

md("""A custom transform participates in the same three calling styles.

{table(header=["style", "result"])}{{ shift_rows }}
""")

**Direct and factory modes.** Use `mode="direct"` when the transform can run as `op(expr)`. Use `mode="factory"` when pipeline use should look like `expr >> op(...)`. Function style still passes the expression first, such as `op(expr, ...)`.

**Activated namespace.** `math_toolkit.notebook.activate()` injects `ExprTransform`, `make_transform_op`, `unhold`, and the built-in transform names directly into the notebook namespace. The original SymPy functions remain available through the `sympy` module, such as `sympy.expand(...)`.

In [9]:
angle_series = make_transform_op(
    lambda expr: expr.series(x, 0, 5).removeO(),
    name="angle_series",
    mode="direct",
)

sin(x) >> angle_series

   3    
  x     
- ── + x
  6     

## Semantics

**Operation objects.** A registered transform is an `ExprTransformOp`. It preserves the wrapped callable metadata, remains callable in ordinary function form, and knows how to receive an expression from the left side of `>>`.

**Stages.** Calling a factory transform without the expression creates an `ExprTransformStage`. For example, `subs({x: 3})` is a stage waiting for the expression that appears on its left.

**Why `>>`.** SymPy already uses `|` for logical and set-like operations, so transform pipelines use `>>` to read as "send this expression through the next step" without taking over SymPy's logical notation.

## Pitfalls

**Factory transforms need their arguments.** Write `expr >> subs({x: 3})`, not `expr >> subs`. The bare factory operation does not know which substitution, variable, or precision you want.

**Remember `.Done()`.** Fluent chains return another `ExprTransform` at every step so chaining can continue. `.Done()` extracts the final expression.

**Use expression-first callables.** `make_transform_op` expects the wrapped callable to accept the expression as its first argument. Choose a unique name when registering a custom fluent method.

**Parenthesize when clarity matters.** Python parses shifts after ordinary arithmetic, but parentheses make longer left-hand expressions easier to scan.

In [10]:
try:
    (x + 1) >> subs
except TypeError as error:
    print(error)

subs requires arguments in pipeline usage. Use expr >> subs(...).


## See also

[Expression](Expression.ipynb)  
[subs](subs.ipynb)  
[rewrite](rewrite.ipynb)  
[composing expressions](../tutorials/composing_expressions.ipynb)